In [2]:
is_causal = True # Set to False for future leakage

In [3]:
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_, spectral_norm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import random
import numpy as np
import matplotlib.pyplot as plt

In [4]:
import sys
sys.path.append("../EDAF-Group_6-PV_Forecasting")

In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
pv_data = pd.read_csv("../data/PV_2022_hourly.csv")
weather_data = pd.read_csv("../data/Weather_clean_NEMS.csv")

In [8]:
df_pv = pd.DataFrame(pv_data)
df_weather = pd.DataFrame(weather_data)

In [9]:
df_pv.info()

<class 'pandas.DataFrame'>
RangeIndex: 8758 entries, 0 to 8757
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TimestampInUtc  8758 non-null   str    
 1   pv              8758 non-null   float64
dtypes: float64(1), str(1)
memory usage: 137.0 KB


In [10]:
weather_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   timestamp                    8760 non-null   str    
 1   Temperature                  8760 non-null   float64
 2   Sunshine Duration            8760 non-null   float64
 3   Shortwave Radiation          8760 non-null   float64
 4   Direct Shortwave Radiation   8760 non-null   float64
 5   Diffuse Shortwave Radiation  8760 non-null   float64
 6   Snowfall Amount              8760 non-null   float64
 7   Relative Humidity            8760 non-null   float64
 8   Cloud Cover Total            8760 non-null   float64
dtypes: float64(8), str(1)
memory usage: 616.1 KB


In [11]:
df_pv.head()

,TimestampInUtc,pv
0,2022-01-01 00:00:00,0.0
1,2022-01-01 01:00:00,0.0
2,2022-01-01 02:00:00,0.0
3,2022-01-01 03:00:00,0.0
4,2022-01-01 04:00:00,0.0


In [12]:
df_weather.head()

,timestamp,Temperature,Sunshine Duration,Shortwave Radiation,Direct Shortwave Radiation,Diffuse Shortwave Radiation,Snowfall Amount,Relative Humidity,Cloud Cover Total
0,2022-01-01 00:00:00,9.657269,0.0,0.0,0.0,0.0,0.0,29.0,0.0
1,2022-01-01 01:00:00,9.457270,0.0,0.0,0.0,0.0,0.0,29.0,0.0
2,2022-01-01 02:00:00,9.787270,0.0,0.0,0.0,0.0,0.0,28.0,0.0
3,2022-01-01 03:00:00,9.467270,0.0,0.0,0.0,0.0,0.0,28.0,0.0
4,2022-01-01 04:00:00,8.797270,0.0,0.0,0.0,0.0,0.0,30.0,0.0


In [13]:
df_pv.shape

(8758, 2)

In [14]:
df_weather.shape

(8760, 9)

In [15]:
df_pv[-10:]

,TimestampInUtc,pv
8748,2022-12-31 12:00:00,50.272210
8749,2022-12-31 13:00:00,31.335803
8750,2022-12-31 14:00:00,6.479349
8751,2022-12-31 15:00:00,0.000000
8752,2022-12-31 16:00:00,0.000000
8753,2022-12-31 17:00:00,0.000000
8754,2022-12-31 18:00:00,0.000000
8755,2022-12-31 19:00:00,0.000000
8756,2022-12-31 20:00:00,0.000000
8757,2022-12-31 21:00:00,0.000000


In [16]:
df_pv["TimestampInUtc"] = pd.to_datetime(df_pv["TimestampInUtc"])

last_time = df_pv["TimestampInUtc"].iloc[-1]

new_rows = pd.DataFrame({
    "TimestampInUtc": [
        last_time + pd.Timedelta(hours=1),
        last_time + pd.Timedelta(hours=2)
    ],
    "pv": [0, 0]
})

df_pv = pd.concat([df_pv, new_rows], ignore_index=True)

In [17]:
df_pv[-10:]

,TimestampInUtc,pv
8750,2022-12-31 14:00:00,6.479349
8751,2022-12-31 15:00:00,0.000000
8752,2022-12-31 16:00:00,0.000000
8753,2022-12-31 17:00:00,0.000000
8754,2022-12-31 18:00:00,0.000000
8755,2022-12-31 19:00:00,0.000000
8756,2022-12-31 20:00:00,0.000000
8757,2022-12-31 21:00:00,0.000000
8758,2022-12-31 22:00:00,0.000000
8759,2022-12-31 23:00:00,0.000000


In [18]:
df_pv.shape

(8760, 2)

In [19]:
df_weather.shape

(8760, 9)

In [20]:
df_merged = pd.concat([df_pv, df_weather], axis=1)

In [21]:
df_merged.head()

,TimestampInUtc,pv,timestamp,Temperature,Sunshine Duration,Shortwave Radiation,Direct Shortwave Radiation,Diffuse Shortwave Radiation,Snowfall Amount,Relative Humidity,Cloud Cover Total
0,2022-01-01 00:00:00,0.0,2022-01-01 00:00:00,9.657269,0.0,0.0,0.0,0.0,0.0,29.0,0.0
1,2022-01-01 01:00:00,0.0,2022-01-01 01:00:00,9.457270,0.0,0.0,0.0,0.0,0.0,29.0,0.0
2,2022-01-01 02:00:00,0.0,2022-01-01 02:00:00,9.787270,0.0,0.0,0.0,0.0,0.0,28.0,0.0
3,2022-01-01 03:00:00,0.0,2022-01-01 03:00:00,9.467270,0.0,0.0,0.0,0.0,0.0,28.0,0.0
4,2022-01-01 04:00:00,0.0,2022-01-01 04:00:00,8.797270,0.0,0.0,0.0,0.0,0.0,30.0,0.0


In [22]:
df_merged.iloc[-10:]

,TimestampInUtc,pv,timestamp,Temperature,Sunshine Duration,Shortwave Radiation,Direct Shortwave Radiation,Diffuse Shortwave Radiation,Snowfall Amount,Relative Humidity,Cloud Cover Total
8750,2022-12-31 14:00:00,6.479349,2022-12-31 14:00:00,15.597270,40.851063,307.05000,175.559000,131.490980,0.0,64.0,30.000002
8751,2022-12-31 15:00:00,0.000000,2022-12-31 15:00:00,14.687269,40.851063,246.53000,138.154100,108.375900,0.0,64.0,30.000002
8752,2022-12-31 16:00:00,0.000000,2022-12-31 16:00:00,13.027269,47.744680,147.73999,80.121185,67.618805,0.0,68.0,19.200000
8753,2022-12-31 17:00:00,0.000000,2022-12-31 17:00:00,11.937269,33.407090,32.04000,16.751293,15.288709,0.0,68.0,30.000002
8754,2022-12-31 18:00:00,0.000000,2022-12-31 18:00:00,12.117270,0.000000,0.00000,0.000000,0.000000,0.0,61.0,18.600000
8755,2022-12-31 19:00:00,0.000000,2022-12-31 19:00:00,11.847270,0.000000,0.00000,0.000000,0.000000,0.0,59.0,100.000000
8756,2022-12-31 20:00:00,0.000000,2022-12-31 20:00:00,11.437269,0.000000,0.00000,0.000000,0.000000,0.0,58.0,95.000000
8757,2022-12-31 21:00:00,0.000000,2022-12-31 21:00:00,10.897269,0.000000,0.00000,0.000000,0.000000,0.0,58.0,20.000000
8758,2022-12-31 22:00:00,0.000000,2022-12-31 22:00:00,10.407269,0.000000,0.00000,0.000000,0.000000,0.0,59.0,10.000000
8759,2022-12-31 23:00:00,0.000000,2022-12-31 23:00:00,10.557270,0.000000,0.00000,0.000000,0.000000,0.0,58.0,26.000000


In [23]:
df_merged.shape

(8760, 11)

In [24]:
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'])
df_merged["hour"] = df_merged["timestamp"].dt.hour
df_merged["day_of_year"] = df_merged["timestamp"].dt.dayofyear
df_merged["month"] = df_merged["timestamp"].dt.month

In [25]:
df_merged.columns

Index(['TimestampInUtc', 'pv', 'timestamp', 'Temperature', 'Sunshine Duration',
       'Shortwave Radiation', 'Direct Shortwave Radiation',
       'Diffuse Shortwave Radiation', 'Snowfall Amount', 'Relative Humidity',
       'Cloud Cover Total', 'hour', 'day_of_year', 'month'],
      dtype='str')

In [26]:
df_merged["hour_sin"]  = np.sin(2 * np.pi * df_merged["hour"] / 24)
df_merged["hour_cos"]  = np.cos(2 * np.pi * df_merged["hour"] / 24)

df_merged["month_sin"] = np.sin(2 * np.pi * df_merged["month"] / 12)
df_merged["month_cos"] = np.cos(2 * np.pi * df_merged["month"] / 12)

df_merged["doy_sin"] = np.sin(2 * np.pi * df_merged["day_of_year"] / 365)
df_merged["doy_cos"] = np.cos(2 * np.pi * df_merged["day_of_year"] / 365)

In [27]:
df_merged.iloc[-744:]

,TimestampInUtc,pv,timestamp,Temperature,Sunshine Duration,Shortwave Radiation,Direct Shortwave Radiation,Diffuse Shortwave Radiation,Snowfall Amount,Relative Humidity,Cloud Cover Total,hour,day_of_year,month,hour_sin,hour_cos,month_sin,month_cos,doy_sin,doy_cos
8016,2022-12-01 00:00:00,0.0,2022-12-01 00:00:00,2.387269,0.0,0.0,0.0,0.0,0.0,90.0,100.0,0,335,12,0.000000,1.000000,-2.449294e-16,1.0,-4.937756e-01,0.869589
8017,2022-12-01 01:00:00,0.0,2022-12-01 01:00:00,2.257269,0.0,0.0,0.0,0.0,0.0,90.0,100.0,1,335,12,0.258819,0.965926,-2.449294e-16,1.0,-4.937756e-01,0.869589
8018,2022-12-01 02:00:00,0.0,2022-12-01 02:00:00,2.057269,0.0,0.0,0.0,0.0,0.0,91.0,100.0,2,335,12,0.500000,0.866025,-2.449294e-16,1.0,-4.937756e-01,0.869589
8019,2022-12-01 03:00:00,0.0,2022-12-01 03:00:00,1.737269,0.0,0.0,0.0,0.0,0.0,93.0,100.0,3,335,12,0.707107,0.707107,-2.449294e-16,1.0,-4.937756e-01,0.869589
8020,2022-12-01 04:00:00,0.0,2022-12-01 04:00:00,1.277269,0.0,0.0,0.0,0.0,0.0,94.0,100.0,4,335,12,0.866025,0.500000,-2.449294e-16,1.0,-4.937756e-01,0.869589
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2022-12-31 19:00:00,0.0,2022-12-31 19:00:00,11.847270,0.0,0.0,0.0,0.0,0.0,59.0,100.0,19,365,12,-0.965926,0.258819,-2.449294e-16,1.0,6.432491e-16,1.000000
8756,2022-12-31 20:00:00,0.0,2022-12-31 20:00:00,11.437269,0.0,0.0,0.0,0.0,0.0,58.0,95.0,20,365,12,-0.866025,0.500000,-2.449294e-16,1.0,6.432491e-16,1.000000
8757,2022-12-31 21:00:00,0.0,2022-12-31 21:00:00,10.897269,0.0,0.0,0.0,0.0,0.0,58.0,20.0,21,365,12,-0.707107,0.707107,-2.449294e-16,1.0,6.432491e-16,1.000000
8758,2022-12-31 22:00:00,0.0,2022-12-31 22:00:00,10.407269,0.0,0.0,0.0,0.0,0.0,59.0,10.0,22,365,12,-0.500000,0.866025,-2.449294e-16,1.0,6.432491e-16,1.000000


In [28]:
df_merged.drop(columns=['TimestampInUtc', 'timestamp', 'hour', 'day_of_year', 'month'], inplace=True)

In [52]:
import pandas as pd

PV_LAGS = [1, 2, 3, 4, 8, 16, 24, 48, 168]

def add_pv_lags(df: pd.DataFrame, pv_col: str = "pv", lags=PV_LAGS) -> pd.DataFrame:
    df_out = df.copy()

    for lag in lags:
        col_name = f"{pv_col}_lag_{lag}"
        df_out[col_name] = df_out[pv_col].shift(lag)

    # drop the first max(lags) rows (they have NaNs because of shifting)
    max_lag = int(max(lags))
    df_out = df_out.iloc[max_lag:].copy()

    return df_out

# usage (do this after your df is sorted by time index!)
df_model_with_lags = add_pv_lags(df_merged, pv_col="pv", lags=PV_LAGS)

print(df_model_with_lags.columns)
print("Rows before:", len(df_merged), "Rows after:", len(df_model_with_lags))


Index(['pv', 'Temperature', 'Sunshine Duration', 'Shortwave Radiation',
       'Direct Shortwave Radiation', 'Diffuse Shortwave Radiation',
       'Snowfall Amount', 'Relative Humidity', 'Cloud Cover Total', 'hour_sin',
       'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'pv_lag_1',
       'pv_lag_2', 'pv_lag_3', 'pv_lag_4', 'pv_lag_8', 'pv_lag_16',
       'pv_lag_24', 'pv_lag_48', 'pv_lag_168'],
      dtype='str')
Rows before: 8760 Rows after: 8592


In [53]:
df_test = df_model_with_lags.iloc[-744:].copy()
df_train = df_model_with_lags.iloc[:-744].copy()

In [57]:
df_test.head(50)

,pv,Temperature,Sunshine Duration,Shortwave Radiation,Direct Shortwave Radiation,Diffuse Shortwave Radiation,Snowfall Amount,Relative Humidity,Cloud Cover Total,hour_sin,...,doy_cos,pv_lag_1,pv_lag_2,pv_lag_3,pv_lag_4,pv_lag_8,pv_lag_16,pv_lag_24,pv_lag_48,pv_lag_168
8016,0.000000,2.387269,0.000000,0.00000,0.000000,0.000000,0.0,90.0,100.0,0.000000e+00,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,5.095247,0.000000,0.000000,0.000000
8017,0.000000,2.257269,0.000000,0.00000,0.000000,0.000000,0.0,90.0,100.0,2.588190e-01,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,7.251804,0.000000,0.000000,0.000000
8018,0.000000,2.057269,0.000000,0.00000,0.000000,0.000000,0.0,91.0,100.0,5.000000e-01,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,12.268580,0.000000,0.000000,0.000000
8019,0.000000,1.737269,0.000000,0.00000,0.000000,0.000000,0.0,93.0,100.0,7.071068e-01,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,9.654091,0.000000,0.000000,0.000000
8020,0.000000,1.277269,0.000000,0.00000,0.000000,0.000000,0.0,94.0,100.0,8.660254e-01,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,3.909768,0.000000,0.000000,0.000000
8021,0.000000,1.047270,0.000000,0.00000,0.000000,0.000000,0.0,94.0,100.0,9.659258e-01,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,1.840292,0.000000,0.000000,0.000000
8022,0.000000,0.817269,0.000000,0.00000,0.000000,0.000000,0.0,94.0,100.0,1.000000e+00,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,0.250038,0.000000,1.278466,8.626852
8023,1.764190,0.557269,0.000000,0.00000,0.000000,0.000000,0.0,95.0,100.0,9.659258e-01,...,0.869589,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.875216,13.536182,29.344423
8024,5.580462,0.297269,0.000000,0.00000,0.000000,0.000000,0.0,96.0,100.0,8.660254e-01,...,0.869589,1.764190,0.000000,0.000000,0.000000,0.000000,0.000000,5.095247,29.441714,22.183903
8025,6.703691,0.617269,0.000000,50.73000,26.782812,23.947187,0.0,94.0,100.0,7.071068e-01,...,0.869589,5.580462,1.764190,0.000000,0.000000,0.000000,0.000000,7.251804,45.583387,45.140680


In [30]:
df_train.columns

Index(['pv', 'Temperature', 'Sunshine Duration', 'Shortwave Radiation',
       'Direct Shortwave Radiation', 'Diffuse Shortwave Radiation',
       'Snowfall Amount', 'Relative Humidity', 'Cloud Cover Total', 'hour_sin',
       'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos'],
      dtype='str')

In [ ]:
TARGET_COL = "pv"

CONTINUOUS_FEATURES = [
    "Temperature", "Sunshine Duration", "Shortwave Radiation",
    "Direct Shortwave Radiation", "Diffuse Shortwave Radiation",
    "Snowfall Amount", "Relative Humidity", "Cloud Cover Total", 
    "pv_lag_1", "pv_lag_2", "pv_lag_3", "pv_lag_4", "pv_lag_8", "pv_lag_16", "pv_lag_24", "pv_lag_48", "pv_lag_168"
]

CYCLIC_FEATURES = [
    "hour_sin", "hour_cos",
    "month_sin", "month_cos",
    "doy_sin", "doy_cos"
]

CATEGORICAL_FEATURES = []

In [32]:
def prepare_data(df_train, df_test):

    # ---------- continuous ----------
    Xc_train = df_train[CONTINUOUS_FEATURES].values
    Xc_test  = df_test[CONTINUOUS_FEATURES].values

    # ---------- cyclic ----------
    Xcy_train = df_train[CYCLIC_FEATURES].values
    Xcy_test  = df_test[CYCLIC_FEATURES].values

    # ---------- target ----------
    y_train = df_train[[TARGET_COL]].values
    y_test  = df_test[[TARGET_COL]].values

    y_train_unscaled = y_train.copy()

    # ---------- scalers ----------
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    # fit on train
    Xc_train_scaled = scaler_X.fit_transform(Xc_train)
    Xc_test_scaled  = scaler_X.transform(Xc_test)

    y_train_scaled = scaler_y.fit_transform(y_train)
    y_test_scaled  = scaler_y.transform(y_test)

    # ---------- combine ----------
    X_train = np.concatenate([Xc_train_scaled, Xcy_train], axis=1).astype(np.float32)
    X_test  = np.concatenate([Xc_test_scaled,  Xcy_test],  axis=1).astype(np.float32)

    y_train = y_train_scaled.astype(np.float32)
    y_test  = y_test_scaled.astype(np.float32)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y,
        "y_train_unscaled": y_train_unscaled
    }

In [33]:
data = prepare_data(df_train, df_test)

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

In [34]:
LOOKBACK = 168   # 7 days (tune: 24, 48, 72, 168)
HORIZON  = 24

In [35]:
def create_sequences_multistep(X, y, lookback, horizon=24):
    Xs, ys = [], []
    last_i = len(X) - lookback - horizon + 1

    for i in range(last_i):
        Xs.append(X[i:i + lookback])  # (lookback, n_features)

        y_window = y[i + lookback : i + lookback + horizon]  # (horizon, 1)
        y_window = y_window[:, 0]                            # (horizon,)
        ys.append(y_window)

    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


In [36]:
X_train_seq, y_train_seq = create_sequences_multistep(X_train, y_train, LOOKBACK, HORIZON)
X_test_seq, y_test_seq = create_sequences_multistep(X_test, y_test, LOOKBACK, HORIZON)

In [37]:
print("Train seq:", X_train_seq.shape, y_train_seq.shape)
print("Test  seq:", X_test_seq.shape,  y_test_seq.shape)

Train seq: (7825, 168, 14) (7825, 24)
Test  seq: (553, 168, 14) (553, 24)


In [38]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [39]:
class Chomp1d(nn.Module):
    """Remove extra padding from causal conv to keep output length same as input"""
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

In [40]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding, dropout, causal):
        super().__init__()
        # Replace weight_norm with spectral_norm
        self.conv1 = spectral_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                              stride=stride, padding=padding, dilation=dilation))
        if causal:
            self.chomp1 = Chomp1d(padding)
        else:
            self.chomp1 = nn.Identity()
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = spectral_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                              stride=stride, padding=padding, dilation=dilation))
        if causal:
            self.chomp2 = Chomp1d(padding)
        else:
            self.chomp2 = nn.Identity()
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

In [41]:
class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            in_ch = input_size if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            dilation = 2**i
            
            if is_causal:
                padding_value = (kernel_size - 1) * dilation
            else:
                # symmetric "same" padding for non-causal
                padding_value = ((kernel_size - 1) * dilation) // 2

            layers.append(
                TemporalBlock(
                    in_ch, out_ch, kernel_size,
                    stride=1, dilation=dilation,
                    padding=padding_value,
                    dropout=dropout,
                    causal=is_causal
                )
            )

        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        # TCN expects (batch, channels, seq_len)
        x = x.transpose(1, 2)  # (B, seq_len, features) → (B, features, seq_len)
        y = self.network(x)
        y = y[:, :, -1]        # take last time step
        y = self.fc(y)
        return y

In [42]:
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        return base_lr * (epoch + 1) / warmup_epochs

In [43]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / n

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)
        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / n

In [ ]:
# # Hyperparams
# BATCH_SIZE = 64
# EPOCHS = 20
# LR = 1e-3

# NUM_CHANNELS = [32, 32, 32, 32]
# KERNEL_SIZE = 3
# DROPOUT = 0.2
# CAUSAL = is_causal

# criterion = nn.MSELoss()

# def run_timeseries_kfold(X_seq, y_seq, n_splits=5):
#     tscv = TimeSeriesSplit(n_splits=n_splits)

#     fold_losses = []
#     for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_seq)):
#         train_ds = SeqDataset(X_seq[tr_idx], y_seq[tr_idx])
#         val_ds   = SeqDataset(X_seq[va_idx], y_seq[va_idx])

#         train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
#         val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

#         model = TCN(
#             input_size=X_seq.shape[-1],
#             output_size=HORIZON,
#             num_channels=NUM_CHANNELS,
#             kernel_size=KERNEL_SIZE,
#             dropout=DROPOUT
#         ).to(device)

#         optimizer = torch.optim.Adam(model.parameters(), lr=LR)

#         best_val = float("inf")
#         best_state = None

#         for epoch in range(EPOCHS):
#             tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
#             va_loss = eval_one_epoch(model, val_loader, criterion)

#             if va_loss < best_val:
#                 best_val = va_loss
#                 best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

#         fold_losses.append(best_val)
#         print(f"Fold {fold+1}/{n_splits} | best val MSE: {best_val:.5f}")

#     return fold_losses

# fold_losses = run_timeseries_kfold(X_train_seq, y_train_seq, n_splits=5)
# print("Fold losses:", fold_losses)
# print("Mean ± std:", np.mean(fold_losses), "±", np.std(fold_losses))


c:\Users\illum\.conda\envs\energy_forecasting\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fold 1/5 | best val MSE: 0.21038


KeyboardInterrupt: 

In [ ]:
# import torch
# from torch.utils.data import DataLoader

# FINAL_EPOCHS = 30  # or whatever you want
# MODEL_PATH = "tcn_pv_multistep.pt"

# train_ds = SeqDataset(X_train_seq, y_train_seq)
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# model = TCN(
#     input_size=X_train_seq.shape[-1],
#     output_size=HORIZON,
#     num_channels=NUM_CHANNELS,
#     kernel_size=KERNEL_SIZE,
#     dropout=DROPOUT
# ).to(device)

# optimizer = torch.optim.Adam(model.parameters(), lr=LR)
# criterion = torch.nn.MSELoss()

# best_loss = float("inf")
# best_state = None

# for epoch in range(FINAL_EPOCHS):
#     tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)

#     if tr_loss < best_loss:
#         best_loss = tr_loss
#         best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

#     print(f"Epoch {epoch+1}/{FINAL_EPOCHS} | train MSE: {tr_loss:.5f}")

# # load best state (optional but recommended)
# if best_state is not None:
#     model.load_state_dict(best_state)

# ckpt = {
#     "model_state": model.state_dict(),
#     "hparams": {
#         "LOOKBACK": LOOKBACK,
#         "HORIZON": HORIZON,
#         "NUM_CHANNELS": NUM_CHANNELS,
#         "KERNEL_SIZE": KERNEL_SIZE,
#         "DROPOUT": DROPOUT,
#     },
#     # important for inverse transform later
#     "scaler_y": data["scaler_y"],
#     "feature_cols": CONTINUOUS_FEATURES + CYCLIC_FEATURES,
# }
# torch.save(ckpt, MODEL_PATH)

# print("Saved:", MODEL_PATH)


Epoch 1/30 | train MSE: 0.65324
Epoch 2/30 | train MSE: 0.31276
Epoch 3/30 | train MSE: 0.26231
Epoch 4/30 | train MSE: 0.24922
Epoch 5/30 | train MSE: 0.23035
Epoch 6/30 | train MSE: 0.21957
Epoch 7/30 | train MSE: 0.21146
Epoch 8/30 | train MSE: 0.20384
Epoch 9/30 | train MSE: 0.19727
Epoch 10/30 | train MSE: 0.19698
Epoch 11/30 | train MSE: 0.19226
Epoch 12/30 | train MSE: 0.18948
Epoch 13/30 | train MSE: 0.18986
Epoch 14/30 | train MSE: 0.18743
Epoch 15/30 | train MSE: 0.18549
Epoch 16/30 | train MSE: 0.18276
Epoch 17/30 | train MSE: 0.18179
Epoch 18/30 | train MSE: 0.18099
Epoch 19/30 | train MSE: 0.17886
Epoch 20/30 | train MSE: 0.17948
Epoch 21/30 | train MSE: 0.17730
Epoch 22/30 | train MSE: 0.17749
Epoch 23/30 | train MSE: 0.17462
Epoch 24/30 | train MSE: 0.17474
Epoch 25/30 | train MSE: 0.17339
Epoch 26/30 | train MSE: 0.17235
Epoch 27/30 | train MSE: 0.17170
Epoch 28/30 | train MSE: 0.17154
Epoch 29/30 | train MSE: 0.16889
Epoch 30/30 | train MSE: 0.16838
Saved: tcn_pv_multi

In [45]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import TimeSeriesSplit

FINAL_EPOCHS = 30
MODEL_PATH = "tcn_pv_best_fold.pt"

def build_model(input_size: int) -> torch.nn.Module:
    model = TCN(
        input_size=input_size,
        output_size=HORIZON,
        num_channels=NUM_CHANNELS,
        kernel_size=KERNEL_SIZE,
        dropout=DROPOUT
    ).to(device)
    return model

def train_with_timeseries_kfold(
    X_seq,
    y_seq,
    n_splits=5,
    epochs=30,
    batch_size=64,
    lr=1e-3,
    model_path="tcn_pv_best_fold.pt"
):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_best_vals = []
    best_overall_val = float("inf")
    best_overall_state = None
    best_overall_fold = None

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_seq)):
        print(f"\n===== Fold {fold+1}/{n_splits} =====")

        train_ds = SeqDataset(X_seq[tr_idx], y_seq[tr_idx])
        val_ds = SeqDataset(X_seq[va_idx], y_seq[va_idx])

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

        model = build_model(input_size=X_seq.shape[-1])
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.MSELoss()

        best_val = float("inf")
        best_state = None

        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            va_loss = eval_one_epoch(model, val_loader, criterion)

            train_losses.append(tr_loss)
            val_losses.append(va_loss)

            improved = va_loss < best_val
            if improved:
                best_val = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            print(
                f"Fold {fold+1} | Epoch {epoch+1}/{epochs} "
                f"| train MSE: {tr_loss:.5f} "
                f"| val MSE: {va_loss:.5f} "
                f"| best val: {best_val:.5f}"
            )

        fold_best_vals.append(best_val)

        is_best_overall = best_val < best_overall_val
        if is_best_overall:
            best_overall_val = best_val
            best_overall_state = best_state
            best_overall_fold = fold + 1

    print("\n===== CV Summary =====")
    print("Fold best val MSEs:", [float(x) for x in fold_best_vals])
    print("Mean ± std:", float(np.mean(fold_best_vals)), "±", float(np.std(fold_best_vals)))
    print("Best overall fold:", best_overall_fold, "| best val MSE:", float(best_overall_val))

    if best_overall_state is not None:
        best_model = build_model(input_size=X_seq.shape[-1])
        best_model.load_state_dict(best_overall_state)

        ckpt = {
            "model_state": best_model.state_dict(),
            "hparams": {
                "LOOKBACK": LOOKBACK,
                "HORIZON": HORIZON,
                "NUM_CHANNELS": NUM_CHANNELS,
                "KERNEL_SIZE": KERNEL_SIZE,
                "DROPOUT": DROPOUT,
            },
            "scaler_y": data["scaler_y"],
            "feature_cols": CONTINUOUS_FEATURES + CYCLIC_FEATURES,
            "cv": {
                "n_splits": n_splits,
                "epochs": epochs,
                "batch_size": batch_size,
                "lr": lr,
                "best_overall_fold": int(best_overall_fold) if best_overall_fold is not None else None,
                "best_overall_val_mse": float(best_overall_val),
                "fold_best_vals": [float(x) for x in fold_best_vals],
            }
        }
        torch.save(ckpt, model_path)
        print("Saved best-fold model:", model_path)

    return fold_best_vals, best_overall_fold


# Run CV training
fold_losses, best_fold = train_with_timeseries_kfold(
    X_train_seq,
    y_train_seq,
    n_splits=5,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    model_path=MODEL_PATH
)



===== Fold 1/5 =====
Fold 1 | Epoch 1/30 | train MSE: 0.52459 | val MSE: 1.01743 | best val: 1.01743
Fold 1 | Epoch 2/30 | train MSE: 0.28971 | val MSE: 0.92499 | best val: 0.92499
Fold 1 | Epoch 3/30 | train MSE: 0.23085 | val MSE: 0.81532 | best val: 0.81532
Fold 1 | Epoch 4/30 | train MSE: 0.19085 | val MSE: 0.71103 | best val: 0.71103
Fold 1 | Epoch 5/30 | train MSE: 0.16227 | val MSE: 0.61115 | best val: 0.61115
Fold 1 | Epoch 6/30 | train MSE: 0.14521 | val MSE: 0.53683 | best val: 0.53683
Fold 1 | Epoch 7/30 | train MSE: 0.12947 | val MSE: 0.46911 | best val: 0.46911
Fold 1 | Epoch 8/30 | train MSE: 0.11716 | val MSE: 0.40310 | best val: 0.40310
Fold 1 | Epoch 9/30 | train MSE: 0.10939 | val MSE: 0.36236 | best val: 0.36236
Fold 1 | Epoch 10/30 | train MSE: 0.10378 | val MSE: 0.35471 | best val: 0.35471
Fold 1 | Epoch 11/30 | train MSE: 0.09289 | val MSE: 0.32455 | best val: 0.32455
Fold 1 | Epoch 12/30 | train MSE: 0.08230 | val MSE: 0.27773 | best val: 0.27773
Fold 1 | Epoch 

In [47]:
MODEL_PATH = "tcn_pv_best_fold.pt"#"tcn_pv_multistep.pt"

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

model = TCN(
    input_size=X_train_seq.shape[-1],
    output_size=ckpt["hparams"]["HORIZON"],
    num_channels=ckpt["hparams"]["NUM_CHANNELS"],
    kernel_size=ckpt["hparams"]["KERNEL_SIZE"],
    dropout=ckpt["hparams"]["DROPOUT"]
).to(device)

model.load_state_dict(ckpt["model_state"])
model.eval()

scaler_y = ckpt["scaler_y"]


In [48]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Predict (scaled) ---
test_X = torch.tensor(X_test_seq, dtype=torch.float32)
test_loader = DataLoader(TensorDataset(test_X), batch_size=256, shuffle=False)

pred_scaled_parts = []

model.eval()
with torch.no_grad():
    for (Xb,) in test_loader:
        Xb = Xb.to(device)
        pb = model(Xb)  # (B, HORIZON) scaled
        pred_scaled_parts.append(pb.cpu().numpy())

pred_scaled = np.concatenate(pred_scaled_parts, axis=0)  # (N, HORIZON)
y_true_scaled = y_test_seq                                # (N, HORIZON)

# --- Scaled metrics (directly comparable to train MSE) ---
scaled_mae = mean_absolute_error(y_true_scaled.reshape(-1), pred_scaled.reshape(-1))
scaled_mse = mean_squared_error(y_true_scaled.reshape(-1), pred_scaled.reshape(-1))
scaled_rmse = np.sqrt(scaled_mse)
scaled_r2 = r2_score(y_true_scaled.reshape(-1), pred_scaled.reshape(-1))

print("=== Scaled metrics (compare with train loss) ===")
print("MAE:", scaled_mae)
print("MSE:", scaled_mse)
print("RMSE:", scaled_rmse)
print("R2:", scaled_r2)

# --- Unscale both y and pred ---
pred_unscaled = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
y_true_unscaled = scaler_y.inverse_transform(y_true_scaled.reshape(-1, 1)).reshape(y_true_scaled.shape)

# --- Unscaled metrics (real PV units) ---
unscaled_mae = mean_absolute_error(y_true_unscaled.reshape(-1), pred_unscaled.reshape(-1))
unscaled_mse = mean_squared_error(y_true_unscaled.reshape(-1), pred_unscaled.reshape(-1))
unscaled_rmse = np.sqrt(unscaled_mse)
unscaled_r2 = r2_score(y_true_unscaled.reshape(-1), pred_unscaled.reshape(-1))

print("\n=== Unscaled metrics (real PV units) ===")
print("MAE:", unscaled_mae)
print("MSE:", unscaled_mse)
print("RMSE:", unscaled_rmse)
print("R2:", unscaled_r2)

# --- Optional: per-horizon metrics (often very revealing) ---
h_mae = []
h_rmse = []

for h in range(HORIZON):
    mae_h = mean_absolute_error(y_true_unscaled[:, h], pred_unscaled[:, h])
    rmse_h = np.sqrt(mean_squared_error(y_true_unscaled[:, h], pred_unscaled[:, h]))
    h_mae.append(mae_h)
    h_rmse.append(rmse_h)

print("\n=== Per-horizon (unscaled) ===")
print("MAE per step:", np.round(h_mae, 3))
print("RMSE per step:", np.round(h_rmse, 3))
print("MAE t+1:", h_mae[0], "| MAE t+24:", h_mae[-1])


=== Scaled metrics (compare with train loss) ===
MAE: 0.169696643948555
MSE: 0.058668382465839386
RMSE: 0.24221557023824744
R2: -1.5733301639556885

=== Unscaled metrics (real PV units) ===
MAE: 15.77173900604248
MSE: 506.77685546875
RMSE: 22.511704854780547
R2: -1.5733301639556885

=== Per-horizon (unscaled) ===
MAE per step: [13.629 14.404 15.708 15.061 16.508 17.746 17.735 17.602 17.478 15.665
 16.659 13.687 14.603 14.96  15.621 15.863 17.73  14.983 19.224 17.247
 15.021 13.613 13.303 14.472]
RMSE per step: [20.59  21.878 23.808 22.054 23.791 25.019 24.613 24.289 23.725 22.156
 22.238 19.816 20.982 20.926 22.513 22.16  24.836 21.232 26.343 23.884
 21.008 19.126 20.338 21.199]
MAE t+1: 13.62925910949707 | MAE t+24: 14.472006797790527


In [50]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

test_X = torch.tensor(X_test_seq, dtype=torch.float32)
test_loader = DataLoader(TensorDataset(test_X), batch_size=256, shuffle=False)

pred_scaled = []

with torch.no_grad():
    for (Xb,) in test_loader:
        Xb = Xb.to(device)
        pb = model(Xb)                  # (B, HORIZON)
        pred_scaled.append(pb.cpu().numpy())

pred_scaled = np.concatenate(pred_scaled, axis=0)  # (N_test_seq, HORIZON)

# inverse transform: scaler expects 2D (n, 1), so reshape
pred_unscaled = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)

print("pred_scaled:", pred_scaled.shape)
print("pred_unscaled:", pred_unscaled.shape)


pred_scaled: (553, 24)
pred_unscaled: (553, 24)


Target std: 92.9407813363149
Train MSE in real units: 1451.1821243479897


In [51]:
import matplotlib.pyplot as plt

i = 0  # pick any index
plt.figure(figsize=(10,4))
plt.plot(y_test_unscaled[i], label="True")
plt.plot(pred_unscaled[i], label="Pred")
plt.title(f"One 24-step forecast (sample {i})")
plt.xlabel("Horizon step")
plt.ylabel("PV")
plt.legend()
plt.show()


NameError: name 'y_test_unscaled' is not defined

<Figure size 1000x400 with 0 Axes>